# Feature Scaling and Selection

## Scaling: Why and When

| Method | Formula | Best For |
|--------|---------|----------|
| **StandardScaler** | $(x - \mu) / \sigma$ | Normally distributed features |
| **MinMaxScaler** | $(x - x_{\min}) / (x_{\max} - x_{\min})$ | Bounded features, neural nets |
| **RobustScaler** | $(x - \text{median}) / \text{IQR}$ | Features with outliers |

**Algorithms that NEED scaling:** KNN, SVM, logistic regression, PCA, neural networks (anything distance or gradient based).  
**Algorithms that DON'T need scaling:** Decision trees, random forests, gradient boosting (split-based).

## Selection: Three Families

| Type | Method | Pros | Cons |
|------|--------|------|------|
| **Filter** | Variance, correlation, MI | Fast, model-agnostic | Ignores feature interactions |
| **Wrapper** | RFE, forward/backward | Considers interactions | Slow, overfitting risk |
| **Embedded** | L1 (Lasso), tree importance | Built into training | Model-specific |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.metrics import accuracy_score

np.random.seed(42)

## Feature Scaling Implementations

In [ ]:
class StandardScaler:
    """Z-score normalization: (x - mean) / std."""
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_[self.std_ == 0] = 1.0
        return self
    
    def transform(self, X):
        return (X - self.mean_) / self.std_

class MinMaxScaler:
    """Scale to [0, 1] range."""
    def fit(self, X):
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        self.range_ = self.max_ - self.min_
        self.range_[self.range_ == 0] = 1.0
        return self
    
    def transform(self, X):
        return (X - self.min_) / self.range_

class RobustScaler:
    """Scale using median and IQR (robust to outliers)."""
    def fit(self, X):
        self.median_ = np.median(X, axis=0)
        q25 = np.percentile(X, 25, axis=0)
        q75 = np.percentile(X, 75, axis=0)
        self.iqr_ = q75 - q25
        self.iqr_[self.iqr_ == 0] = 1.0
        return self
    
    def transform(self, X):
        return (X - self.median_) / self.iqr_

# Create data with different scales
n = 1000
X_raw = np.column_stack([
    np.random.normal(100, 50, n),       # Feature 0: large scale
    np.random.normal(0.5, 0.1, n),      # Feature 1: small scale
    np.random.exponential(5, n),         # Feature 2: skewed with outliers
    np.random.uniform(0, 1, n),          # Feature 3: uniform [0,1]
])

# Add some outliers to feature 2
X_raw[np.random.choice(n, 10), 2] = np.random.uniform(50, 100, 10)

# Scale
ss = StandardScaler().fit(X_raw)
X_standard = ss.transform(X_raw)

mm = MinMaxScaler().fit(X_raw)
X_minmax = mm.transform(X_raw)

rs = RobustScaler().fit(X_raw)
X_robust = rs.transform(X_raw)

print("Original ranges:")
for i in range(4):
    print(f"  Feature {i}: [{X_raw[:,i].min():.2f}, {X_raw[:,i].max():.2f}], mean={X_raw[:,i].mean():.2f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

titles = ['Original', 'StandardScaler', 'MinMaxScaler', 'RobustScaler']
data_list = [X_raw, X_standard, X_minmax, X_robust]
feature_names = ['Large scale', 'Small scale', 'Skewed+outliers', 'Uniform']

for ax, title, data in zip(axes.flat, titles, data_list):
    bp = ax.boxplot([data[:, i] for i in range(4)], labels=feature_names, patch_artist=True)
    for patch, color in zip(bp['boxes'], ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']):
        patch.set_facecolor(color)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Effect of Different Scaling Methods', fontsize=14)
plt.tight_layout()
plt.show()

## Demo: Which Algorithms Need Scaling?

In [ ]:
# Create a classification problem with features at very different scales
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           random_state=42)

# Artificially create different scales
scales = [1, 100, 0.01, 1000, 0.001, 1, 50, 0.1, 500, 1]
X_scaled_diff = X * scales

X_train, X_test, y_train, y_test = train_test_split(X_scaled_diff, y, test_size=0.2, random_state=42)

# StandardScaler applied
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)': SVC(kernel='rbf'),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=5),
}

print(f"{'Model':<25} {'No scaling':<15} {'With scaling':<15} {'Difference'}")
print("-" * 70)

for name, model in models.items():
    # Without scaling
    m1 = type(model)(**model.get_params())
    m1.fit(X_train, y_train)
    acc_no = accuracy_score(y_test, m1.predict(X_test))
    
    # With scaling
    m2 = type(model)(**model.get_params())
    m2.fit(X_train_s, y_train)
    acc_yes = accuracy_score(y_test, m2.predict(X_test_s))
    
    diff = acc_yes - acc_no
    marker = '***' if abs(diff) > 0.05 else ''
    print(f"{name:<25} {acc_no:<15.4f} {acc_yes:<15.4f} {diff:+.4f} {marker}")

print("\n*** = large difference -- scaling matters for this algorithm")

## Feature Selection: Filter Methods

### Variance Threshold
Remove features with variance below a threshold. Features with near-zero variance carry no information.

In [ ]:
# Generate data with some useless features
X, y = make_classification(n_samples=500, n_features=20, n_informative=8,
                           n_redundant=4, n_repeated=2, random_state=42)

# Add near-constant features
X = np.column_stack([
    X,
    np.ones(500) * 5 + np.random.randn(500) * 0.001,  # near-constant
    np.random.choice([0, 0, 0, 0, 1], 500),            # very low variance
])

feature_names = [f'f{i}' for i in range(X.shape[1])]
print(f"Total features: {X.shape[1]}")

def variance_threshold(X, threshold=0.01):
    """Remove features with variance below threshold."""
    variances = X.var(axis=0)
    mask = variances > threshold
    return X[:, mask], mask, variances

X_var, mask, variances = variance_threshold(X, threshold=0.1)
removed = [feature_names[i] for i in range(len(mask)) if not mask[i]]
print(f"Features removed (var < 0.1): {removed}")
print(f"Features remaining: {X_var.shape[1]}")

### Correlation Filter
Remove one of each pair of highly correlated features (they carry redundant information).

In [ ]:
def correlation_filter(X, threshold=0.9):
    """Remove one feature from each highly correlated pair."""
    corr = np.corrcoef(X.T)
    n_features = X.shape[1]
    to_remove = set()
    
    for i in range(n_features):
        if i in to_remove:
            continue
        for j in range(i + 1, n_features):
            if j in to_remove:
                continue
            if abs(corr[i, j]) > threshold:
                to_remove.add(j)  # Remove the later one
    
    mask = [i not in to_remove for i in range(n_features)]
    return X[:, mask], mask, corr

X_corr, mask_corr, corr_matrix = correlation_filter(X, threshold=0.85)
removed_corr = [feature_names[i] for i in range(len(mask_corr)) if not mask_corr[i]]
print(f"Correlated features removed: {removed_corr}")
print(f"Features remaining: {X_corr.shape[1]}")

# Visualize correlation matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix[:20, :20], cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix')
plt.colorbar(im)
plt.tight_layout()
plt.show()

### Mutual Information Based Selection

Mutual information measures the dependency between a feature and the target, capturing non-linear relationships that correlation misses.

$$MI(X; Y) = \sum_{x,y} p(x,y) \log \frac{p(x,y)}{p(x)p(y)}$$

MI = 0 means X and Y are independent. Higher MI = more useful feature.

In [ ]:
def mutual_information(X, y, n_bins=10):
    """Estimate mutual information between each feature and target using binning."""
    mi_scores = []
    
    for j in range(X.shape[1]):
        # Discretize continuous feature
        x_binned = np.digitize(X[:, j], bins=np.linspace(X[:, j].min(), X[:, j].max(), n_bins))
        
        # Compute joint and marginal distributions
        n = len(y)
        mi = 0.0
        
        for x_val in np.unique(x_binned):
            for y_val in np.unique(y):
                p_xy = np.mean((x_binned == x_val) & (y == y_val))
                p_x = np.mean(x_binned == x_val)
                p_y = np.mean(y == y_val)
                
                if p_xy > 0 and p_x > 0 and p_y > 0:
                    mi += p_xy * np.log(p_xy / (p_x * p_y))
        
        mi_scores.append(mi)
    
    return np.array(mi_scores)

mi_scores = mutual_information(X[:, :20], y)  # First 20 original features

# Visualize
fig, ax = plt.subplots(figsize=(12, 4))
names_20 = feature_names[:20]
order = np.argsort(mi_scores)[::-1]
ax.bar(range(20), mi_scores[order], color='steelblue')
ax.set_xticks(range(20))
ax.set_xticklabels([names_20[i] for i in order], rotation=45)
ax.set_ylabel('Mutual Information')
ax.set_title('Feature Importance by Mutual Information')
ax.axhline(y=0.02, color='red', linestyle='--', label='Threshold')
ax.legend()
plt.tight_layout()
plt.show()

selected = [names_20[i] for i in range(20) if mi_scores[i] > 0.02]
print(f"Selected features (MI > 0.02): {selected}")

## Feature Selection: Embedded Method -- L1 (Lasso)

L1 regularization drives feature weights exactly to zero, performing automatic feature selection.

$$\min_w \|y - Xw\|^2 + \alpha \|w\|_1$$

Features with $w_j = 0$ after training are effectively removed.

In [ ]:
from sklearn.linear_model import LassoCV

# Use first 20 features for clarity
X_sel = X[:, :20]
X_train_sel, X_test_sel, y_train, y_test = train_test_split(X_sel, y, test_size=0.2, random_state=42)

# Standardize first (important for L1)
scaler = StandardScaler().fit(X_train_sel)
X_train_s = scaler.transform(X_train_sel)
X_test_s = scaler.transform(X_test_sel)

# Fit Lasso with cross-validation to pick alpha
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train_s, y_train)

print(f"Best alpha: {lasso.alpha_:.6f}")
print(f"\nFeature weights:")
for i, (name, coef) in enumerate(zip(feature_names[:20], lasso.coef_)):
    status = 'SELECTED' if abs(coef) > 1e-6 else 'removed'
    print(f"  {name}: {coef:+.4f}  [{status}]")

n_selected = np.sum(np.abs(lasso.coef_) > 1e-6)
print(f"\nFeatures selected by L1: {n_selected}/{20}")

## Feature Selection: Wrapper Method -- Recursive Feature Elimination

In [ ]:
def recursive_feature_elimination(X_train, y_train, X_test, y_test, n_select=8):
    """RFE: iteratively remove least important feature."""
    n_features = X_train.shape[1]
    active = list(range(n_features))
    elimination_order = []
    scores = []
    
    while len(active) > n_select:
        # Train model on active features
        model = LogisticRegression(max_iter=1000)
        model.fit(X_train[:, active], y_train)
        
        # Score
        acc = accuracy_score(y_test, model.predict(X_test[:, active]))
        scores.append((len(active), acc))
        
        # Find least important feature (smallest absolute coefficient)
        importances = np.abs(model.coef_[0])
        worst_idx = np.argmin(importances)
        worst_feature = active[worst_idx]
        
        elimination_order.append(feature_names[worst_feature])
        active.pop(worst_idx)
    
    # Final score
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train[:, active], y_train)
    acc = accuracy_score(y_test, model.predict(X_test[:, active]))
    scores.append((len(active), acc))
    
    return active, elimination_order, scores

selected_features, elim_order, rfe_scores = recursive_feature_elimination(
    X_train_s, y_train, X_test_s, y_test, n_select=8)

print(f"Selected features: {[feature_names[i] for i in selected_features]}")
print(f"\nElimination order (first removed = least important):")
for i, name in enumerate(elim_order):
    print(f"  {i+1}. {name}")

In [ ]:
# Visualize RFE performance curve
fig, ax = plt.subplots(figsize=(10, 4))
n_feats, accs = zip(*rfe_scores)
ax.plot(n_feats, accs, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Number of Features')
ax.set_ylabel('Test Accuracy')
ax.set_title('RFE: Accuracy vs Number of Features')
ax.invert_xaxis()
ax.grid(True, alpha=0.3)

best_idx = np.argmax(accs)
ax.axvline(x=n_feats[best_idx], color='red', linestyle='--',
           label=f'Best: {n_feats[best_idx]} features, acc={accs[best_idx]:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

## Effect of Scaling on Different Algorithms -- Visualization

In [ ]:
from sklearn.datasets import make_moons

# Create 2D data for visualization, then scale features differently
X_2d, y_2d = make_moons(n_samples=300, noise=0.2, random_state=42)
# Make one feature 100x larger
X_2d_skewed = X_2d.copy()
X_2d_skewed[:, 1] *= 100

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

models_2d = [
    ('KNN', KNeighborsClassifier(n_neighbors=5)),
    ('SVM', SVC(kernel='rbf')),
    ('DecisionTree', DecisionTreeClassifier(max_depth=5)),
]

for col, (name, model) in enumerate(models_2d):
    for row, (data, label) in enumerate([
        (X_2d_skewed, 'Unscaled'),
        (StandardScaler().fit(X_2d_skewed).transform(X_2d_skewed), 'Scaled')
    ]):
        ax = axes[row, col]
        m = type(model)(**model.get_params())
        m.fit(data, y_2d)
        acc = accuracy_score(y_2d, m.predict(data))
        
        # Plot decision boundary
        h = 0.1 if row == 0 else 0.05
        x_min, x_max = data[:, 0].min() - 0.5, data[:, 0].max() + 0.5
        y_min, y_max = data[:, 1].min() - 0.5, data[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                             np.arange(y_min, y_max, h))
        Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        
        ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
        ax.scatter(data[:, 0], data[:, 1], c=y_2d, cmap='RdBu', edgecolors='k', s=20)
        ax.set_title(f'{name} ({label})\nacc={acc:.3f}')

plt.suptitle('Decision Boundaries: Scaling Impact', fontsize=14)
plt.tight_layout()
plt.show()

## Interview Questions

### Q: When is scaling necessary?
**A:** When the algorithm uses distances (KNN, SVM, K-Means) or gradients (neural nets, logistic regression). Without scaling, features with larger magnitudes dominate. Tree-based methods (RF, GBDT) don't need scaling because they use split thresholds, not distances.

### Q: L1 for feature selection -- how?
**A:** L1 regularization adds $\alpha|w_j|$ penalty. Unlike L2 ($\alpha w_j^2$), L1's subdifferential includes zero, so the optimization can push weights exactly to zero. Features with zero weights are effectively removed. Geometrically: L1's diamond-shaped constraint region has corners on axes, making axis-aligned solutions (sparse) likely.

### Q: Filter vs wrapper vs embedded methods?
**A:**
- **Filter** (variance, correlation, MI): Score features independently, no model training. Fast but misses interactions.
- **Wrapper** (RFE, forward selection): Train model repeatedly with different feature subsets. Slow but captures interactions.
- **Embedded** (L1, tree importance): Feature selection happens during model training. Best of both worlds but model-specific.

**In practice:** Start with filter methods to remove obvious noise, then use embedded methods. Use wrappers for final tuning if compute allows.

### Q: StandardScaler vs RobustScaler?
**A:** StandardScaler uses mean/std which are sensitive to outliers. RobustScaler uses median/IQR, so a few extreme values don't distort the scaling. Use RobustScaler when data has outliers you want to keep (not remove).